In [ ]:
# =============================================================================
# IMPORTS
# =============================================================================
# sqlalchemy + snowflake connector: lets Python talk to Snowflake via SQL
# pandas (pd): data manipulation library — think of it like Excel in code
# numpy (np): math library for arrays, NaN handling, log transforms
# IsolationForest: the anomaly detection model — finds "unusual" hospitals
# StandardScaler: normalizes features to mean=0, stdev=1 so no single
#                 feature dominates just because it has bigger numbers
# matplotlib (plt): charting library
# shap: explains WHY the model flagged each hospital
# =============================================================================
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import shap

In [ ]:
# =============================================================================
# SNOWFLAKE CONNECTION
# =============================================================================
# Creates a reusable connection "engine" to Snowflake.
# All pd.read_sql() calls below will use this engine to run queries.
# We only dispose() it after all queries are done (after df_2026 pull).
# =============================================================================
engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

In [ ]:
# =============================================================================
# 2025 DATA PULL
# =============================================================================
# Pulls one row per hospital_group × fin_market for 2025 (Jan-Dec).
# Counts distinct cases that meet each KPI condition.
#
# Filters:
#   - M&R brand, LOC-eligible (loc_flag=1)
#   - FFS population (COSMOS non-institutional + NICE FFS/Physician)
#   - IP types: Medical/Surgical/Transplant (with valid admit date) + LTAC/SNF/AIR
#   - Joined to tin_collection to get hospital_group name from TIN
#   - HAVING >= 30 cases ensures minimum volume for stable rates
#
# Does NOT include member_appeal (only available from 2026 onward).
# =============================================================================
df_2025 = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202501' and '202512'
        and d.collection is not null
    group by 1,2
    having count(distinct a.case_id) >= 30
""", engine)

print(f"df_2025: {df_2025.shape}")

In [ ]:
# =============================================================================
# 2025: COMPUTE RATES + EMPIRICAL BAYES SHRINKAGE + LOG-VOLUME
# =============================================================================
#
# STEP 1: Raw rates (numerator / denominator)
#   - initial_adr_rate = ADR cases / total cases
#   - p2p_overturn_rate = P2P overturns / ADR cases (denominator is ADRs, not total)
#
# STEP 2: Eligibility filter
#   - Keeps only hospitals with >= 30 cases AND >= 10 ADRs
#   - Small hospitals produce noisy rates (e.g., 1/3 = 33% is not meaningful)
#
# STEP 3: Empirical Bayes shrinkage (credibility weighting)
#   Formula: adjusted_rate = (observed_numerator + k * global_rate) / (denominator + k)
#   - k = 50 is the "shrinkage strength" (higher k = more pull toward global)
#   - gr_* = global rates (population averages across ALL eligible hospitals)
#   - Effect: hospitals with few cases get pulled toward the population mean;
#     hospitals with many cases stay close to their observed rate
#   - Example: Hospital A has 2 P2P overturns out of 5 ADRs = raw 40%.
#     With k=50 and global P2P rate=15%: adjusted = (2 + 50*0.15) / (5 + 50) = 15.6%
#     → much more realistic than 40% from just 5 observations
#
# STEP 4: Log-volume features
#   - log1p(x) = natural log of (x + 1). The +1 avoids log(0) errors.
#   - Purpose: tells the IF model "how big is this hospital" alongside the rates.
#     Without this, IF can't distinguish a rate based on 500 cases from one based on 31.
#   - We only log the DENOMINATORS (case_count, initial_adr_cnt) — not numerators,
#     because numerator info is already captured in the rate.
# =============================================================================

# --- Step 1: raw rates ---
df_2025 = df_2025.assign(
    initial_adr_rate = lambda x: x.initial_adr_cnt / x.case_count,
    persistent_adr_rate = lambda x: x.persistent_adr_cnt / x.case_count,
    p2p_overturn_rate = lambda x: x.p2p_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
    appeal_overturn_rate = lambda x: x.appeal_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
    mcr_overturn_rate = lambda x: x.mcr_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
    md_review_rate = lambda x: x.md_reviewed_cnt / x.case_count,
)

rate_cols_25 = [
    "initial_adr_rate", "persistent_adr_rate", "p2p_overturn_rate",
    "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate"
]
count_cols_25 = [
    "case_count", "initial_adr_cnt", "persistent_adr_cnt",
    "p2p_ovrtn_cnt", "appeal_ovrtn_cnt", "mcr_ovrtn_cnt", "md_reviewed_cnt"
]
# Replace infinity and NaN with 0 (happens when denominator is 0)
df_2025[rate_cols_25 + count_cols_25] = df_2025[rate_cols_25 + count_cols_25].replace([np.inf, -np.inf], np.nan).fillna(0)

# --- Step 2: eligibility filter ---
df_iso_25 = df_2025.query("case_count >= 30 and initial_adr_cnt >= 10").copy()

# --- Step 3: Bayes shrinkage ---
k = 50  # shrinkage strength

# Global rates = sum(numerator) / sum(denominator) across all eligible hospitals
gr_initial_adr = df_iso_25["initial_adr_cnt"].sum() / df_iso_25["case_count"].sum()
gr_persistent = df_iso_25["persistent_adr_cnt"].sum() / df_iso_25["case_count"].sum()
gr_p2p = df_iso_25["p2p_ovrtn_cnt"].sum() / df_iso_25["initial_adr_cnt"].sum()
gr_appeal = df_iso_25["appeal_ovrtn_cnt"].sum() / df_iso_25["initial_adr_cnt"].sum()
gr_mcr = df_iso_25["mcr_ovrtn_cnt"].sum() / df_iso_25["initial_adr_cnt"].sum()
gr_md = df_iso_25["md_reviewed_cnt"].sum() / df_iso_25["case_count"].sum()

# Overwrite raw rates with shrinkage-adjusted versions
df_iso_25 = df_iso_25.assign(
    initial_adr_rate = lambda x: (x.initial_adr_cnt + k * gr_initial_adr) / (x.case_count + k),
    persistent_adr_rate = lambda x: (x.persistent_adr_cnt + k * gr_persistent) / (x.case_count + k),
    p2p_overturn_rate = lambda x: (x.p2p_ovrtn_cnt + k * gr_p2p) / (x.initial_adr_cnt + k),
    appeal_overturn_rate = lambda x: (x.appeal_ovrtn_cnt + k * gr_appeal) / (x.initial_adr_cnt + k),
    mcr_overturn_rate = lambda x: (x.mcr_ovrtn_cnt + k * gr_mcr) / (x.initial_adr_cnt + k),
    md_review_rate = lambda x: (x.md_reviewed_cnt + k * gr_md) / (x.case_count + k),
    log_case_count = lambda x: np.log1p(x.case_count),
    log_initial_adr_cnt = lambda x: np.log1p(x.initial_adr_cnt),
)

# --- The feature list that goes into IF ---
iso_kpi_25 = [
    "log_case_count", "log_initial_adr_cnt",
    "initial_adr_rate", "persistent_adr_rate", "p2p_overturn_rate",
    "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate"
]
df_iso_25[iso_kpi_25] = df_iso_25[iso_kpi_25].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"2025 IF-ready: {df_iso_25.shape[0]} hospitals, {len(iso_kpi_25)} features")

In [ ]:
# =============================================================================
# 2025: ISOLATION FOREST
# =============================================================================
#
# StandardScaler: normalizes each feature to mean=0, stdev=1.
#   Without this, a feature like log_case_count (range 3-9) would be dwarfed by
#   a feature like initial_adr_rate (range 0.01-0.50) in terms of variance.
#   Scaling ensures IF treats all features equally.
#
# IsolationForest:
#   - Builds 300 random decision trees (n_estimators=300)
#   - Each tree randomly picks a feature and a split point, trying to isolate
#     each hospital into its own leaf as fast as possible
#   - Anomalies are points that get isolated QUICKLY (short path = few splits)
#     because their feature values are far from the crowd
#   - contamination=0.05: tells the model "expect ~5% of hospitals to be anomalous"
#   - random_state=42: makes results reproducible (same answer every run)
#
# decision_function(X): returns the anomaly score
#   - Negative = anomalous (short average path length)
#   - Positive = normal (long average path length)
#   - More negative = more anomalous
#
# predict(X): returns 1 (normal) or -1 (anomaly)
#   - We convert to True/False with == -1
#
# anomaly_rank: rank 1 = most anomalous, rank N = least anomalous
# =============================================================================

# Scale the features
scaler_25 = StandardScaler()
X_25 = scaler_25.fit_transform(df_iso_25[iso_kpi_25])

# Train the Isolation Forest
iso_model_25 = IsolationForest(contamination = 0.05, n_estimators = 300, random_state = 42)
iso_model_25.fit(X_25)

# Score every hospital
df_iso_25 = df_iso_25.assign(
    anomaly_score = iso_model_25.decision_function(X_25),  # continuous score
    anomaly_flag = iso_model_25.predict(X_25) == -1,       # True if anomalous
)
# Rank: 1 = most anomalous (lowest score)
df_iso_25["anomaly_rank"] = df_iso_25["anomaly_score"].rank(method = "dense", ascending = True).astype(int)

print(f"2025 Flagged: {df_iso_25['anomaly_flag'].sum()} / {df_iso_25.shape[0]} hospitals")

In [ ]:
# =============================================================================
# 2025: SHAP EXPLAINER
# =============================================================================
#
# TreeExplainer: analyzes the IF model's internal tree structure to compute
#   how much each feature contributed to each hospital's anomaly score.
#
# explainer.expected_value = E[f(x)]:
#   The average path length across all hospitals. This is the "baseline" —
#   what the model predicts for a completely average hospital.
#
# shap_values = explainer.shap_values(X_25):
#   Returns a matrix of shape (n_hospitals × n_features).
#   Each cell = "how much did this feature push THIS hospital's path length
#   away from the baseline?"
#   - Negative SHAP value = pushed toward SHORTER path = more anomalous
#   - Positive SHAP value = pushed toward LONGER path = more normal
#
# Key property: base_value + sum(shap_values[i]) = f(x_i) for hospital i
#   (the SHAP values perfectly decompose the model's output)
# =============================================================================
explainer_25 = shap.TreeExplainer(iso_model_25)
shap_values_25 = explainer_25.shap_values(X_25)

print(f"2025 SHAP values shape: {shap_values_25.shape}")

In [ ]:
# =============================================================================
# 2025: SHAP PLOTS
# =============================================================================
#
# LEFT (Beeswarm/Summary): Each dot = one hospital. Shows ALL hospitals.
#   - X-axis = SHAP value (how much this feature pushed the score)
#   - Y-axis = features ranked by importance
#   - Color = actual feature value (red=high, blue=low)
#   - Interpretation: if red dots cluster left → high values of that feature
#     push hospitals toward anomaly
#
# RIGHT (Bar): Mean |SHAP value| per feature across ALL hospitals.
#   - Taller bar = the IF model relies on this feature more heavily
#     when deciding who's normal vs anomalous
#   - This is GLOBAL model behavior, not specific to any one hospital
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize = (16, 5))

plt.sca(axes[0])
shap.summary_plot(shap_values_25, X_25, feature_names = iso_kpi_25, show = False)
axes[0].set_title("2025: SHAP Summary")

plt.sca(axes[1])
shap.summary_plot(shap_values_25, X_25, feature_names = iso_kpi_25, plot_type = "bar", show = False)
axes[1].set_title("2025: Mean Feature Importance")

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 2025: SHAP FLAGGED HOSPITAL DRIVERS
# =============================================================================
#
# For each FLAGGED hospital only, we look at its SHAP values and find:
#   - shap_driver: which feature had the MOST NEGATIVE shap value
#     (= biggest push toward anomaly). This is the "top reason" it was flagged.
#   - shap_driver_value: the actual SHAP contribution (e.g., -2.31 means
#     this feature shortened the path length by 2.31 units)
#   - driver_actual_value: the hospital's actual rate/count for that feature
#     (e.g., if driver is p2p_overturn_rate, this shows the actual 0.42 rate)
#
# idxmin(axis=1): for each row, find the column name with the smallest value
#   → gives us the feature name with most negative SHAP contribution
# =============================================================================
flagged_idx_25 = df_iso_25[df_iso_25["anomaly_flag"]].index
flagged_pos_25 = [df_iso_25.index.get_loc(idx) for idx in flagged_idx_25]

# Build a DataFrame of SHAP values for flagged hospitals only
shap_flagged_25 = pd.DataFrame(
    shap_values_25[flagged_pos_25],
    columns = iso_kpi_25,
    index = flagged_idx_25
)
shap_flagged_25["hospital_group"] = df_iso_25.loc[flagged_idx_25, "hospital_group"].values
shap_flagged_25["fin_market"] = df_iso_25.loc[flagged_idx_25, "fin_market"].values
shap_flagged_25["anomaly_rank"] = df_iso_25.loc[flagged_idx_25, "anomaly_rank"].values

# Find the feature with the most negative SHAP value (biggest push toward anomaly)
shap_flagged_25["shap_driver"] = shap_flagged_25[iso_kpi_25].idxmin(axis = 1)
shap_flagged_25["shap_driver_value"] = shap_flagged_25[iso_kpi_25].min(axis = 1)

# Look up the hospital's actual value for that driver feature
shap_flagged_25["driver_actual_value"] = shap_flagged_25.apply(
    lambda r: df_iso_25.loc[r.name, r["shap_driver"]], axis = 1
)

print(f"2025 Flagged: {len(shap_flagged_25)} hospitals")
shap_flagged_25[["hospital_group", "fin_market", "anomaly_rank", "shap_driver", "shap_driver_value", "driver_actual_value"]].sort_values("anomaly_rank")

In [ ]:
# =============================================================================
# 2025: SHAP WATERFALL — TOP 5 MOST ANOMALOUS
# =============================================================================
#
# For each of the 5 most anomalous hospitals, shows a waterfall chart:
#   - Starts at E[f(x)] (base value = average path length for a typical hospital)
#   - Each bar = one feature's SHAP contribution
#   - Blue bars push LEFT (shorter path → more anomalous)
#   - Red bars push RIGHT (longer path → more normal)
#   - Ends at f(x) = this hospital's actual path length
#
# If f(x) << E[f(x)], the hospital was isolated very quickly → very anomalous.
#
# Note: SHAP works on path length (higher = more normal), while anomaly_score
# from decision_function is the opposite (lower = more anomalous). The waterfall
# uses the raw path length scale.
# =============================================================================

# Get the 5 hospitals with the lowest anomaly scores (most anomalous)
top5_idx_25 = (
    df_iso_25[df_iso_25["anomaly_flag"] & df_iso_25["hospital_group"].notna()]
    .nsmallest(5, "anomaly_score")
    .index
)

# base_val = E[f(x)] = expected path length for an average hospital
base_val_25 = float(explainer_25.expected_value) if np.ndim(explainer_25.expected_value) == 0 else float(explainer_25.expected_value[0])

for idx in top5_idx_25:
    pos = df_iso_25.index.get_loc(idx)
    hospital_name = df_iso_25.loc[idx, "hospital_group"]
    rank = int(df_iso_25.loc[idx, "anomaly_rank"])

    print(f"\n{'='*60}")
    print(f"{hospital_name} | Rank #{rank} (2025)")
    print(f"{'='*60}")

    # Create a SHAP Explanation object for this single hospital
    explanation = shap.Explanation(
        values = shap_values_25[pos],      # this hospital's SHAP values
        base_values = base_val_25,          # E[f(x)]
        feature_names = iso_kpi_25
    )
    shap.waterfall_plot(explanation, show = False)
    plt.title(f"{hospital_name[:35]} | Rank #{rank} (2025)")
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# 2025: OUTPUT ASSEMBLY
# =============================================================================
#
# Builds the final output table for 2025 with:
#   - driver: the SHAP-identified top reason (feature name)
#   - direction: "high" or "low" — is the driver above or below the median?
#   - reason: combined string like "p2p_overturn_rate (high)"
#   - driver_value: the hospital's actual value for the driver feature
#   - driver_pop_median: the population median for that feature
#   - driver_dev: % deviation from median = (value - median) / median
#
# For log-volume drivers (log_case_count, log_initial_adr_cnt), we report
# the RAW count (not the log) for human readability.
# =============================================================================

# Assign SHAP driver to all rows (non-flagged get empty string)
df_iso_25["driver"] = shap_flagged_25["shap_driver"].reindex(df_iso_25.index).fillna("")

# Determine direction: is the driver value above or below population median?
df_iso_25["direction"] = np.where(
    df_iso_25["anomaly_flag"],
    df_iso_25.apply(lambda r: "high" if r["driver"] and df_iso_25.loc[r.name, r["driver"]] > df_iso_25[r["driver"]].median() else "low", axis=1),
    ""
)
df_iso_25["reason"] = np.where(df_iso_25["anomaly_flag"], df_iso_25["driver"] + " (" + df_iso_25["direction"] + ")", "")

# Population medians for deviation calculation
pop_medians_25 = df_iso_25[iso_kpi_25].median()
raw_med_case = df_iso_25["case_count"].median()
raw_med_adr = df_iso_25["initial_adr_cnt"].median()

df_iso_25["driver_value"] = np.nan
df_iso_25["driver_pop_median"] = np.nan
df_iso_25["driver_dev"] = np.nan

# For log drivers → report the raw count (more interpretable)
mask = df_iso_25["anomaly_flag"] & (df_iso_25["driver"] == "log_case_count")
df_iso_25.loc[mask, "driver_value"] = df_iso_25.loc[mask, "case_count"]
df_iso_25.loc[mask, "driver_pop_median"] = raw_med_case
df_iso_25.loc[mask, "driver_dev"] = (df_iso_25.loc[mask, "case_count"] - raw_med_case) / raw_med_case

mask = df_iso_25["anomaly_flag"] & (df_iso_25["driver"] == "log_initial_adr_cnt")
df_iso_25.loc[mask, "driver_value"] = df_iso_25.loc[mask, "initial_adr_cnt"]
df_iso_25.loc[mask, "driver_pop_median"] = raw_med_adr
df_iso_25.loc[mask, "driver_dev"] = (df_iso_25.loc[mask, "initial_adr_cnt"] - raw_med_adr) / raw_med_adr

# For rate drivers → report the rate value directly
for rc in rate_cols_25:
    mask = df_iso_25["anomaly_flag"] & (df_iso_25["driver"] == rc)
    if mask.any():
        df_iso_25.loc[mask, "driver_value"] = df_iso_25.loc[mask, rc]
        df_iso_25.loc[mask, "driver_pop_median"] = pop_medians_25[rc]
        df_iso_25.loc[mask, "driver_dev"] = (df_iso_25.loc[mask, rc] - pop_medians_25[rc]) / pop_medians_25[rc]

# Assemble final columns and sort by anomaly_score (most anomalous first)
output_cols_25 = [
    "hospital_group", "fin_market",
    *count_cols_25, *iso_kpi_25,
    "anomaly_score", "anomaly_rank", "anomaly_flag",
    "driver", "direction", "reason",
    "driver_value", "driver_pop_median", "driver_dev"
]
df_iso_output_25 = df_iso_25[output_cols_25].sort_values("anomaly_score").reset_index(drop = True)
df_iso_output_25["year"] = "2025"

print(f"2025 output: {df_iso_output_25.shape[0]} rows | Flagged: {df_iso_output_25['anomaly_flag'].sum()}")
df_iso_output_25[df_iso_output_25["anomaly_flag"]]

In [ ]:
# =============================================================================
# 2026 DATA PULL
# =============================================================================
# Same structure as 2025 but:
#   - Filters to hce_admit_month >= '202601' (2026 onward)
#   - HAVING >= 10 initial ADRs (lower threshold because fewer months of data)
#   - Includes member_appeal_cnt and member_appeal_ovtn_cnt (new KPI for 2026)
# =============================================================================
df_2026 = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
        , count(distinct case when a.member_appeal_ind = 1 then a.case_id end) as member_appeal_cnt
        , count(distinct case when a.member_appeal_ovtn_ind = 1 then a.case_id end) as member_appeal_ovtn_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand = 'M&R'
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month >= '202601'
        and d.collection is not null
    group by 1,2
    having count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) >= 10
""", engine)

engine.dispose()  # Done with both pulls — close the connection
print(f"df_2026: {df_2026.shape}")

In [ ]:
# =============================================================================
# 2026: COMPUTE RATES + BAYES SHRINKAGE + LOG-VOLUME
# =============================================================================
# Same logic as 2025 but adds member_appeal_rate:
#   member_appeal_rate = member_appeal_cnt / initial_adr_cnt
#   (denominator is ADRs because members can only appeal after a denial)
#
# 2026 has 9 IF features vs 2025's 8 (the extra is member_appeal_rate)
# =============================================================================
df_2026 = df_2026.assign(
    initial_adr_rate = lambda x: x.initial_adr_cnt / x.case_count,
    persistent_adr_rate = lambda x: x.persistent_adr_cnt / x.case_count,
    p2p_overturn_rate = lambda x: x.p2p_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
    appeal_overturn_rate = lambda x: x.appeal_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
    mcr_overturn_rate = lambda x: x.mcr_ovrtn_cnt / x.initial_adr_cnt.replace(0, np.nan),
    md_review_rate = lambda x: x.md_reviewed_cnt / x.case_count,
    member_appeal_rate = lambda x: x.member_appeal_cnt / x.initial_adr_cnt.replace(0, np.nan),
)

rate_cols_26 = [
    "initial_adr_rate", "persistent_adr_rate", "p2p_overturn_rate",
    "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate",
    "member_appeal_rate"
]
count_cols_26 = [
    "case_count", "initial_adr_cnt", "persistent_adr_cnt",
    "p2p_ovrtn_cnt", "appeal_ovrtn_cnt", "mcr_ovrtn_cnt", "md_reviewed_cnt",
    "member_appeal_cnt", "member_appeal_ovtn_cnt"
]
df_2026[rate_cols_26 + count_cols_26] = df_2026[rate_cols_26 + count_cols_26].replace([np.inf, -np.inf], np.nan).fillna(0)

df_iso_26 = df_2026.query("case_count >= 30 and initial_adr_cnt >= 10").copy()

k = 50
gr_initial_adr = df_iso_26["initial_adr_cnt"].sum() / df_iso_26["case_count"].sum()
gr_persistent = df_iso_26["persistent_adr_cnt"].sum() / df_iso_26["case_count"].sum()
gr_p2p = df_iso_26["p2p_ovrtn_cnt"].sum() / df_iso_26["initial_adr_cnt"].sum()
gr_appeal = df_iso_26["appeal_ovrtn_cnt"].sum() / df_iso_26["initial_adr_cnt"].sum()
gr_mcr = df_iso_26["mcr_ovrtn_cnt"].sum() / df_iso_26["initial_adr_cnt"].sum()
gr_md = df_iso_26["md_reviewed_cnt"].sum() / df_iso_26["case_count"].sum()
gr_member_appeal = df_iso_26["member_appeal_cnt"].sum() / df_iso_26["initial_adr_cnt"].sum()

df_iso_26 = df_iso_26.assign(
    initial_adr_rate = lambda x: (x.initial_adr_cnt + k * gr_initial_adr) / (x.case_count + k),
    persistent_adr_rate = lambda x: (x.persistent_adr_cnt + k * gr_persistent) / (x.case_count + k),
    p2p_overturn_rate = lambda x: (x.p2p_ovrtn_cnt + k * gr_p2p) / (x.initial_adr_cnt + k),
    appeal_overturn_rate = lambda x: (x.appeal_ovrtn_cnt + k * gr_appeal) / (x.initial_adr_cnt + k),
    mcr_overturn_rate = lambda x: (x.mcr_ovrtn_cnt + k * gr_mcr) / (x.initial_adr_cnt + k),
    md_review_rate = lambda x: (x.md_reviewed_cnt + k * gr_md) / (x.case_count + k),
    member_appeal_rate = lambda x: (x.member_appeal_cnt + k * gr_member_appeal) / (x.initial_adr_cnt + k),
    log_case_count = lambda x: np.log1p(x.case_count),
    log_initial_adr_cnt = lambda x: np.log1p(x.initial_adr_cnt),
)

iso_kpi_26 = [
    "log_case_count", "log_initial_adr_cnt",
    "initial_adr_rate", "persistent_adr_rate", "p2p_overturn_rate",
    "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate",
    "member_appeal_rate"
]
df_iso_26[iso_kpi_26] = df_iso_26[iso_kpi_26].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"2026 IF-ready: {df_iso_26.shape[0]} hospitals, {len(iso_kpi_26)} features")

In [ ]:
# =============================================================================
# 2026: ISOLATION FOREST (same approach as 2025)
# =============================================================================
scaler_26 = StandardScaler()
X_26 = scaler_26.fit_transform(df_iso_26[iso_kpi_26])

iso_model_26 = IsolationForest(contamination = 0.05, n_estimators = 300, random_state = 42)
iso_model_26.fit(X_26)

df_iso_26 = df_iso_26.assign(
    anomaly_score = iso_model_26.decision_function(X_26),
    anomaly_flag = iso_model_26.predict(X_26) == -1,
)
df_iso_26["anomaly_rank"] = df_iso_26["anomaly_score"].rank(method = "dense", ascending = True).astype(int)

print(f"2026 Flagged: {df_iso_26['anomaly_flag'].sum()} / {df_iso_26.shape[0]} hospitals")

In [ ]:
# =============================================================================
# 2026: SHAP EXPLAINER (same approach as 2025)
# =============================================================================
explainer_26 = shap.TreeExplainer(iso_model_26)
shap_values_26 = explainer_26.shap_values(X_26)

print(f"2026 SHAP values shape: {shap_values_26.shape}")

In [ ]:
# =============================================================================
# 2026: SHAP PLOTS (same approach as 2025)
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize = (16, 5))

plt.sca(axes[0])
shap.summary_plot(shap_values_26, X_26, feature_names = iso_kpi_26, show = False)
axes[0].set_title("2026: SHAP Summary")

plt.sca(axes[1])
shap.summary_plot(shap_values_26, X_26, feature_names = iso_kpi_26, plot_type = "bar", show = False)
axes[1].set_title("2026: Mean Feature Importance")

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 2026: SHAP FLAGGED HOSPITAL DRIVERS (same approach as 2025)
# =============================================================================
flagged_idx_26 = df_iso_26[df_iso_26["anomaly_flag"]].index
flagged_pos_26 = [df_iso_26.index.get_loc(idx) for idx in flagged_idx_26]

shap_flagged_26 = pd.DataFrame(
    shap_values_26[flagged_pos_26],
    columns = iso_kpi_26,
    index = flagged_idx_26
)
shap_flagged_26["hospital_group"] = df_iso_26.loc[flagged_idx_26, "hospital_group"].values
shap_flagged_26["fin_market"] = df_iso_26.loc[flagged_idx_26, "fin_market"].values
shap_flagged_26["anomaly_rank"] = df_iso_26.loc[flagged_idx_26, "anomaly_rank"].values
shap_flagged_26["shap_driver"] = shap_flagged_26[iso_kpi_26].idxmin(axis = 1)
shap_flagged_26["shap_driver_value"] = shap_flagged_26[iso_kpi_26].min(axis = 1)
shap_flagged_26["driver_actual_value"] = shap_flagged_26.apply(
    lambda r: df_iso_26.loc[r.name, r["shap_driver"]], axis = 1
)

print(f"2026 Flagged: {len(shap_flagged_26)} hospitals")
shap_flagged_26[["hospital_group", "fin_market", "anomaly_rank", "shap_driver", "shap_driver_value", "driver_actual_value"]].sort_values("anomaly_rank")

In [ ]:
# =============================================================================
# 2026: SHAP WATERFALL — TOP 5 (same approach as 2025)
# =============================================================================
top5_idx_26 = (
    df_iso_26[df_iso_26["anomaly_flag"] & df_iso_26["hospital_group"].notna()]
    .nsmallest(5, "anomaly_score")
    .index
)

base_val_26 = float(explainer_26.expected_value) if np.ndim(explainer_26.expected_value) == 0 else float(explainer_26.expected_value[0])

for idx in top5_idx_26:
    pos = df_iso_26.index.get_loc(idx)
    hospital_name = df_iso_26.loc[idx, "hospital_group"]
    rank = int(df_iso_26.loc[idx, "anomaly_rank"])

    print(f"\n{'='*60}")
    print(f"{hospital_name} | Rank #{rank} (2026)")
    print(f"{'='*60}")

    explanation = shap.Explanation(
        values = shap_values_26[pos],
        base_values = base_val_26,
        feature_names = iso_kpi_26
    )
    shap.waterfall_plot(explanation, show = False)
    plt.title(f"{hospital_name[:35]} | Rank #{rank} (2026)")
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# STACK 2025 + 2026 AND WRITE TO SNOWFLAKE
# =============================================================================
#
# pd.concat: stacks the two DataFrames vertically (like UNION ALL in SQL)
# ignore_index=True: resets row numbers to 0, 1, 2, ...
#
# The YEAR column lets you filter in downstream queries:
#   SELECT * FROM ... WHERE YEAR = '2026' AND ANOMALY_FLAG = TRUE
#
# Columns are uppercased because Snowflake expects uppercase identifiers.
# to_sql() with if_exists='replace' drops and recreates the table each run.
# =============================================================================
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL

engine_write = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

df_stacked = pd.concat([df_iso_output_25, df_iso_output_26], ignore_index = True)
df_stacked.columns = [c.upper() for c in df_stacked.columns]

df_stacked.to_sql(
    "kn_loc_if_outlier_hospitals_mnr_2025_2026",
    engine_write,
    schema = "TMP_1M",
    if_exists = "replace",
    index = False
)

engine_write.dispose()

flagged_25 = ((df_stacked['YEAR'] == '2025') & (df_stacked['ANOMALY_FLAG'] == True)).sum()
flagged_26 = ((df_stacked['YEAR'] == '2026') & (df_stacked['ANOMALY_FLAG'] == True)).sum()
print(f"Wrote {len(df_stacked)} rows to TMP_1M.KN_LOC_IF_OUTLIER_HOSPITALS_MNR_2025_2026")
print(f"  2025: {(df_stacked['YEAR'] == '2025').sum()} rows ({flagged_25} flagged)")
print(f"  2026: {(df_stacked['YEAR'] == '2026').sum()} rows ({flagged_26} flagged)")
df_stacked[df_stacked['ANOMALY_FLAG'] == True][['HOSPITAL_GROUP', 'FIN_MARKET', 'YEAR', 'ANOMALY_RANK', 'DRIVER', 'REASON']]